In [1]:
# === CELL 1: STABLE INSTALLATION ===
import os
import subprocess

print("⏳ Installing Stable Unsloth & Dependencies...")

# 1. Install Unsloth and Unsloth-Zoo directly from PyPI (Guarantees they are synced)
subprocess.run('pip install --upgrade --no-cache-dir unsloth unsloth-zoo', shell=True)

# 2. Install the exact Hugging Face dependencies Unsloth expects
subprocess.run('pip install --no-deps xformers trl peft accelerate bitsandbytes', shell=True)

print("✅ Installation Complete. You can proceed directly to Cell 2!")

⏳ Installing Stable Unsloth & Dependencies...
✅ Installation Complete. You can proceed directly to Cell 2!


In [2]:
# === CELL 2: TRAIN & SAVE (A100 OPTIMIZED) ===
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ================= CONFIGURATION =================
# 1. YOUR DATASET
dataset_path = "train.jsonl"

# 2. TRAINING MODE (Final Run Settings)
TRAINING_STEPS = -1      # -1 tells the trainer to use NUM_EPOCHS instead
SAMPLE_SIZE    = None    # MUST BE None! This ensures all 16,000 balanced rows are used.
NUM_EPOCHS     = 3       # The industry standard for fine-tuning convergence
# =================================================

# ── 1. Load Model ──
print("Loading Qwen 2.5 7B (4-bit)...")
max_seq_length = 4096
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/qwen2.5-7b-bnb-4bit", # <-- UPDATED TO QWEN
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# ── 2. LoRA Adapters ──
# Increased r=32 for maximum learning capacity on the complex JSON task.
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# ── 3. Prompt Template ──
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    texts = []
    for inst, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = alpaca_prompt.format(inst, inp, out) + tokenizer.eos_token
        texts.append(text)
    return {"text": texts}

# ── 4. Load & Prepare Data ──
dataset = load_dataset("json", data_files=dataset_path, split="train")

if SAMPLE_SIZE is not None:
    dataset = dataset.select(range(min(SAMPLE_SIZE, len(dataset))))

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Training on {len(dataset)} examples.")

# ── 5. Training Arguments ──
training_args = TrainingArguments(
    per_device_train_batch_size = 2,     # Increased to feed the A100 faster
    gradient_accumulation_steps = 4,     # Keeps effective batch size at 8
    warmup_ratio       = 0.1,
    max_steps          = TRAINING_STEPS,
    num_train_epochs   = NUM_EPOCHS,
    learning_rate      = 1e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps      = 1,
    optim              = "paged_adamw_8bit", # Paged optimizer as an extra safety net
    weight_decay       = 0.01,
    lr_scheduler_type  = "cosine",
    seed               = 3407,
    output_dir         = "outputs",
    dataloader_num_workers = 0,
)

# ── 6. Train ──
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = training_args,
)

torch.cuda.empty_cache()
trainer.train()

# ── 7. Save GGUF ──
print("Converting to GGUF (q4_k_m)...")
model.save_pretrained_gguf("thesis_final_model", tokenizer, quantization_method = "q4_k_m")
print("DONE! Proceed to download.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Qwen 2.5 7B (4-bit)...
==((====))==  Unsloth 2026.3.3: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:02<00:00, 15.58s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:56<00:00, 14.17s/it]


Unsloth: Merge process complete. Saved to `/content/thesis_final_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['thesis_final_model_gguf/Qwen2.5-7B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['thesis_final_model_gguf/Qwen2.5-7B.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/Qwen2.5-7B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model thesis_final_model_gguf/Qwen2.5-7B.Q4_K_M.gguf -p "why is the sky blue?"
DONE! Proceed to download.


In [4]:
# === CELL 3: SAVE TO GOOGLE DRIVE (The Safe Way) ===
from google.colab import drive
import shutil
import os
import glob

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Define Paths
gguf_folder = "thesis_final_model_gguf"
destination_folder = "/content/drive/My Drive/Thesis_Backup"

# 3. Create Backup Folder
if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)

# 4. Automatically find the GGUF file inside the folder
gguf_files = glob.glob(f"{gguf_folder}/*.gguf")

if not gguf_files:
    print(f"❌ Error: No .gguf file found in {gguf_folder}!")
else:
    source_path = gguf_files[0] # Grab the newly generated model
    # Rename it cleanly for your project
    destination_path = os.path.join(destination_folder, "thesis_model_qwen.gguf")

    print(f"🚚 Copying model {source_path} to Google Drive: {destination_path}...")
    shutil.copy(source_path, destination_path)
    print("✅ SAVED! You can now close this tab safely.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚚 Copying model thesis_final_model_gguf/Qwen2.5-7B.Q4_K_M.gguf to Google Drive: /content/drive/My Drive/Thesis_Backup/thesis_model_qwen.gguf...
✅ SAVED! You can now close this tab safely.
